## Agentic RAG (Tool Based)

In [1]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

### Document Loader

In [2]:
def load_documents(data_folder):
    documents = []
    ids = []
    
    for file in os.listdir(data_folder):
        path = os.path.join(data_folder, file)
        with open(path, 'r') as f:
            text = f.read()
        
        documents.append(text)
        ids.append(file)
    return documents, ids

### Create LLM

In [3]:
llm = ChatOpenAI(api_key=os.getenv('OPENAI_SECRET_KEY'), model='gpt-4o-mini')
embedding_model = OpenAIEmbeddings(api_key=os.getenv('OPENAI_SECRET_KEY'), model='text-embedding-3-small')

llm, embedding_model

(ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000226B26397D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000226B2C5ACD0>, root_client=<openai.OpenAI object at 0x00000226B210A150>, root_async_client=<openai.AsyncOpenAI object at 0x00000226B272D590>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True),
 OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x00000226B2C5B210>, async_clien

### Create VectorDB

In [5]:
data_folder = '../data'

documents, ids = load_documents(data_folder=data_folder)

documents, ids

(['AcmeTech Solutions Code of Conduct\n\nEmployees must maintain professional behavior at all times.\n\nHarassment, discrimination, or unethical conduct will not be tolerated.',
  'AcmeTech Solutions IT Security Policy\n\nEmployees must not share their passwords with anyone.\n\nAll systems must be locked when unattended.\n\nSensitive data must be stored securely and encrypted where appropriate.',
  'AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.',
  "AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employee's manager.\n\nEmployees must ensure a secure and productive work environment while working remotely."],
 ['code_of_conduct.txt',
  'it_security_policy.txt',
  'leave_policy.txt'

In [6]:
vector_db = Chroma.from_texts(
    texts=documents,
    embedding=embedding_model,
    ids=ids,
    persist_directory='../agentic_rag_db',
    collection_metadata={'hnsw:space': 'cosine'}
)

retriever = vector_db.as_retriever(search_kwargs={'k': 2})

vector_db, retriever

(<langchain_community.vectorstores.chroma.Chroma at 0x226b26df590>,
 VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000226B26DF590>, search_kwargs={'k': 2}))

In [7]:
retriever.invoke('work from home')

[Document(metadata={}, page_content="AcmeTech Solutions Remote Work Policy\n\nEmployees are allowed to work remotely up to 2 days per week.\n\nRemote work must be approved by the employee's manager.\n\nEmployees must ensure a secure and productive work environment while working remotely."),
 Document(metadata={}, page_content='AcmeTech Solutions Leave Policy\n\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\n\nLeave requests must be submitted at least two weeks in advance.\n\nUnused leave may not be carried over to the next year unless approved.')]

### Tool

In [10]:
def document_retriever(query):
    docs = retriever.invoke(query)
    context = '\n'.join([doc.page_content for doc in docs])
    return context

### Agentic RAG

In [12]:
def agentic_rag(query):
    print('\n User Query: ', query)
    
    # Initial Retrieval
    context = document_retriever(query)
    print('\nInitial Context: ', context)
    
    # Agent Decision
    decision_prompt = f'''You are an intelligent assistant.
Question:
{query}

Context:
{context}

Do you have enough information (context) to answer the query?
Answer only YES or NO'''

    decision = llm.invoke(decision_prompt).content.strip()
    print('\nAgent Decision: ', decision)
    
    if "no" in decision.lower():
        refine_prompt = f'''Rewrite the given query to retrieve better/more relevant information.
Original Query:
{query}
'''

        refined_query = llm.invoke(refine_prompt).content
        print('\nRefined Query: ', refined_query)
        
        context = document_retriever(refined_query)
        print('\nUpdated Context: ',context)
        
    # Final Answer (Generation)
    final_prompt = f'''Answer the following query using the given context only. 
If the given context is not enough or relevant, then say you don't have enough information to answer the query. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Query:
{query}

Context:
{context}
'''

    answer = llm.invoke(final_prompt).content
    print('\nAnswer: ', answer)
    
    return answer